In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_product_performance
# PURPOSE    : Revenue and performance by product category
# SOURCE     : silver.order_items, silver.orders, silver.products
# DESTINATION: fincompliance_catalog.gold.product_performance
# METRICS:
#   - Total revenue per category
#   - Total units sold per category
#   - Average price per category
#   - Average freight per category
#   - Category ranking by revenue
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    avg,
    round as spark_round,
    rank,
    current_timestamp
)
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CELL 5 - READ FROM SILVER
# ============================================================

df_order_items = spark.table(f"{SILVER_DB}.order_items")
df_orders = spark.table(f"{SILVER_DB}.orders")
df_products = spark.table(f"{SILVER_DB}.products")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"order_items : {df_order_items.count():,} rows")
print(f"orders      : {df_orders.count():,} rows")
print(f"products    : {df_products.count():,} rows")

print("\norder_items columns:")
for c in df_order_items.columns:
    print(f"  {c}")

print("\nproducts columns:")
for c in df_products.columns:
    print(f"  {c}")

In [0]:
# ============================================================
# CALCULATE PRODUCT CATEGORY PERFORMANCE
# Join order_items with orders and products
# Filter delivered orders only
# Group by English category name
# ============================================================

df_product_performance = (
    df_order_items
    .join(
        df_orders.filter(col("order_status") == "delivered")
                  .select("order_id"),
        on="order_id",
        how="inner"
    )
    .join(
        df_products.select(
            "product_id",
            "product_category_name_english",
            "product_size",
            "product_weight_class"
        ),
        on="product_id",
        how="left"
    )
    .groupBy("product_category_name_english")
    .agg(
        count("order_id").alias("total_orders"),
        spark_sum("order_item_id").alias("total_units_sold"),
        spark_round(spark_sum("price"), 2)
            .alias("total_revenue"),
        spark_round(avg("price"), 2)
            .alias("avg_price"),
        spark_round(avg("freight_value"), 2)
            .alias("avg_freight"),
        spark_round(avg("freight_percentage"), 2)
            .alias("avg_freight_pct")
    )
    .orderBy(col("total_revenue").desc())
)

print("Product category performance:")
df_product_performance.show(20, truncate=False)

print(f"\nTotal categories : {df_product_performance.count()}")

In [0]:
# ============================================================
# ADD CATEGORY RANKING
# Rank categories by total revenue
# Useful for Power BI top N visuals
# ============================================================

window_rank = Window.orderBy(col("total_revenue").desc())

df_product_performance = df_product_performance \
    .withColumn(
        "revenue_rank",
        rank().over(window_rank)
    )

print("Top 10 categories with rank:")
df_product_performance \
    .select(
        "revenue_rank",
        "product_category_name_english",
        "total_revenue",
        "total_orders"
    ) \
    .orderBy("revenue_rank") \
    .show(10, truncate=False)

print("Done - Revenue rank added")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_product_performance = df_product_performance \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_product_performance.columns:
    print(f"  {col_name}")
print(f"Total categories : {df_product_performance.count()}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_product_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.product_performance")
)

print(f"Written to {GOLD_DB}.product_performance")
print(f"Rows : {df_product_performance.count()}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.PRODUCT_PERFORMANCE VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.product_performance")
print(f"Total categories : {df_gold.count()}")
print(f"Total columns    : {len(df_gold.columns)}")

print("\nTop 5 categories by revenue:")
df_gold.orderBy("revenue_rank").show(5, truncate=False)

print("\nTotal revenue across all categories:")
total_rev = df_gold.agg(
    spark_round(spark_sum("total_revenue"), 2)
).collect()[0][0]
print(f"  R$ {total_rev:,.2f}")

print("=" * 60)
print("gold.product_performance verification complete!")
print("=" * 60)